# Week 8 — naive scan vs a suffix tree

Open in VS Code. **Select Kernel** (Python 3) in the top right. Then **Run All** from the top.

The code already runs. Read the comments, change one input if you want, watch the printout. Then answer the questions.

Text $t=\mathtt{ATGGTCGGT}$, pattern $p=\mathtt{GGT}$. Hits at $2$ and $6$.

In [ ]:
t = "ATGGTCGGT"
p = "GGT"


def naive(t, p):
    n, m = len(p), len(t)
    hits = []
    comps = 0
    for i in range(0, m - n + 1):
        ok = True
        for k in range(n):
            comps += 1
            if t[i + k] != p[k]:
                ok = False
                break
        if ok:
            hits.append(i)
    return hits, comps


hits, comps = naive(t, p)
print("hits", hits, "  character tests", comps)
print("windows", len(t) - len(p) + 1)
# Play: naive(t, "GGG"), naive(t, "AAAAT")


## Many patterns at once: the keyword tree

Put the patterns into one tree (one letter per edge). From each start position, walk the text's letters down the tree. Count the letters read, and compare with four separate naive scans.

In [ ]:
patterns = ["GGT", "GGG", "ATG", "CG"]

kw = {}
for q in patterns:
    node = kw
    for c in q:
        node = node.setdefault(c, {})
    node["end"] = q

letters = 0
for i in range(len(t)):
    node, j = kw, i
    while j < len(t) and any(k != "end" for k in node):
        letters += 1
        if t[j] not in node:
            break
        node = node[t[j]]
        j += 1
        if "end" in node:
            print("start", i, "hit", node["end"])
print("keyword tree:", letters, "letters read")                          # 18
print("four naive scans:", sum(naive(t, q)[1] for q in patterns))       # 42

## The suffix tree

Insert every suffix of $t\$$ into a tree, one letter per edge, then merge chains of single children into one edge. Each leaf stores where its suffix starts.

In [ ]:
def suffix_tree(t):
    t = t + "$"
    root = {}
    for i in range(len(t)):            # insert suffix t[i:]
        node = root
        for c in t[i:]:
            node = node.setdefault(c, {})
        node["leaf"] = i
    return compress(root)


def compress(node):
    out = {}
    for label, child in node.items():
        if label == "leaf":
            out["leaf"] = child
            continue
        # follow a chain of single children and merge the labels
        while len(child) == 1 and "leaf" not in child:
            (l2, c2), = child.items()
            label, child = label + l2, c2
        out[label] = compress(child)
    return out


def show_tree(node, indent=""):
    for label in sorted(k for k in node if k != "leaf"):
        child = node[label]
        tag = f"   leaf {child['leaf']}" if set(child) == {"leaf"} else ""
        print(f"{indent}{label}{tag}")
        show_tree(child, indent + "    ")


tree = suffix_tree(t)
show_tree(tree)

## Suffix links: letters saved

Insert the suffixes longest first. Suffix $i$ branches off after the longest beginning it shares with an earlier suffix. From the root, that many letters are read. With a suffix link, the walk starts where suffix $i-1$ branched, minus its first letter, so those letters are skipped.

In [ ]:
s = t + "$"


def shared_start(i):
    """Longest beginning of suffix i that also begins an earlier suffix."""
    best = 0
    for j in range(i):
        k = 0
        while s[i + k] == s[j + k]:
            k += 1
        best = max(best, k)
    return best


prev = 0
for i in range(len(s)):
    h = shared_start(i)
    with_link = h - max(prev - 1, 0)
    print(f"suffix {i} {s[i:]:11s} branches after {s[i:i + h] or '(root)':5s}  from root: {h}  with links: {with_link}")
    prev = h

## Search the tree

Read the pattern from the root. A missing edge means the pattern does not occur; otherwise every leaf below is a hit.

In [ ]:
def leaves(node):
    if set(node) == {"leaf"}:
        return [node["leaf"]]
    out = [node["leaf"]] if "leaf" in node else []
    for k, child in node.items():
        if k != "leaf":
            out += leaves(child)
    return out


def search(tree, p):
    node, i, letters = tree, 0, 0
    while i < len(p):
        label = next((k for k in node if k != "leaf" and k[0] == p[i]), None)
        if label is None:
            return [], letters
        for c in label:
            if i == len(p):
                break
            letters += 1
            if c != p[i]:
                return [], letters
            i += 1
        node = node[label]
    return sorted(leaves(node)), letters


for q in ("GGT", "GGGA", "T"):
    hits, letters = search(tree, q)
    print(f"{q:5s} hits {hits}   letters read {letters}")

## How big is the tree?

Count the nodes of the suffix tree with one letter per edge, and after compressing. Try longer random texts.

In [ ]:
import random


def count_nodes(root):
    count, todo = 0, [root]
    while todo:                       # a list instead of recursion: the trees are deep
        node = todo.pop()
        count += 1
        todo += [c for k, c in node.items() if k != "leaf"]
    return count


def one_letter_tree(t):
    t = t + "$"
    root = {}
    for i in range(len(t)):
        node = root
        for c in t[i:]:
            node = node.setdefault(c, {})
        node["leaf"] = i
    return root


print(count_nodes(one_letter_tree(t)), count_nodes(tree))          # 48 15
random.seed(1)
for m in (100, 1000):
    r = "".join(random.choice("ACGT") for _ in range(m))
    print(m + 1, "letters:", count_nodes(one_letter_tree(r)), "nodes before,", count_nodes(suffix_tree(r)), "after compressing")

## Repeats and shared stretches

A string that occurs twice ends at an internal node. The deepest internal node is the longest repeat. For two genomes, build one tree of both and look for the deepest node with leaves from each.

In [ ]:
def deepest(node, label="", want=None):
    """Longest path label of an internal node; want(leaves) filters which nodes count."""
    best = ""
    for k, child in node.items():
        if k == "leaf" or set(child) == {"leaf"}:
            continue
        here = label + k
        if want is None or want(leaves(child)):
            if len(here) > len(best):
                best = here
        deeper = deepest(child, here, want)
        if len(deeper) > len(best):
            best = deeper
    return best


print("longest repeat:", deepest(tree))                          # GGT

t2 = "TTCGGA"
both = {}
for text, end, tag in ((t, "#", 1), (t2, "$", 2)):
    s = text + end
    for i in range(len(s)):
        node = both
        for c in s[i:]:
            node = node.setdefault(c, {})
        node["leaf"] = (tag, i)
both = compress(both)
shared = deepest(both, want=lambda ls: {g for g, _ in ls} == {1, 2})
print("longest shared stretch:", shared)                         # TCGG

## The suffix array

Sort the suffixes and keep only their start positions. Search by binary search: compare the pattern with the middle row, keep the half where it must be.

In [ ]:
s = t + "$"
sa = sorted(range(len(s)), key=lambda i: s[i:].replace("$", " "))   # $ sorts first
for row, i in enumerate(sa):
    print(row, i, s[i:])


def sa_search(p):
    lo, hi = 0, len(sa)
    while lo < hi:                       # first row whose suffix is not smaller than p
        mid = (lo + hi) // 2
        print("  compare with row", mid, s[sa[mid]:])
        if s[sa[mid]:sa[mid] + len(p)].replace("$", " ") < p:
            lo = mid + 1
        else:
            hi = mid
    hits = []
    while lo < len(sa) and s[sa[lo]:].startswith(p):
        hits.append(sa[lo])
        lo += 1
    return hits


print(sa_search("GGT"))     # [6, 2]
print(sa_search("GGGA"))    # []

## Questions

Without looking at notes. **Q1–Q2:** today’s idea. **Q3:** a situation we did not practise.

**Q1.** Why does the naive scan take time $O(nm)$? Where is most of the time spent?

A1.

**Q2.** Once the suffix tree of $t$ is built, why does searching for $p$ not depend on the length of $t$? How do you read off the number of occurrences?

A2.

**Q3.** In the suffix array of $\mathtt{ATGGTCGGT\$}$, compare each row with the next one. How does that give you the longest repeat, without any tree? What is it?

A3.